# Experiment Performance Analysis

Visualizes pass/fail distributions across Langfuse dataset runs (experiments).

## Prerequisites
- Langfuse running (default: `http://localhost:3000`)
- `.env` at workspace root with `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and optionally `EVAL_LANGFUSE_BASE_URL`
- At least one eval run pushed to Langfuse via `uv run eval-run --langfuse`

## Usage
Change `DATASET_NAME` below to analyze a different skill's experiments.
Run names correspond to `--run-name` values passed to `eval-run` (e.g. `scaffold-baseline`, `v2-prompt`).

In [23]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load env vars FIRST, then override LANGFUSE_BASE_URL before importing langfuse
load_dotenv(Path("../.env"))
os.environ["LANGFUSE_BASE_URL"] = (
    os.environ.get("EVAL_LANGFUSE_BASE_URL", "").strip() or "http://localhost:3000"
)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from langfuse import Langfuse

# --- Configuration ---
DATASET_NAME = "evals/DPN Resume Entry Drafter"
# ---------------------

In [24]:
langfuse = Langfuse()
print(f"Connected to Langfuse at {os.environ['LANGFUSE_BASE_URL']}")

Connected to Langfuse at http://localhost:3000


In [25]:
# Fetch all runs for the dataset
runs_response = langfuse.get_dataset_runs(dataset_name=DATASET_NAME)
runs = runs_response.data
print(f"Found {len(runs)} experiment run(s) for '{DATASET_NAME}'")
for r in runs:
    print(f"  - {r.name} (created: {r.created_at})")

Found 2 experiment run(s) for 'evals/DPN Resume Entry Drafter'
  - 2_metric_and_skill_fix (created: 2026-05-22 00:06:00.402000+00:00)
  - 1_scaffold_baseline (created: 2026-05-21 06:02:50.483000+00:00)


In [26]:
# Fetch verdict scores for each run
records = []

for run in runs:
    run_detail = langfuse.get_dataset_run(
        dataset_name=DATASET_NAME, run_name=run.name
    )
    for item in run_detail.dataset_run_items:
        scores = langfuse.api.scores.get_many(
            trace_id=item.trace_id, name="verdict", limit=1
        )
        verdict = None
        if scores.data:
            verdict = bool(scores.data[0].value)
        records.append({
            "run_name": run.name,
            "created_at": run.created_at,
            "trace_id": item.trace_id,
            "dataset_item_id": item.dataset_item_id,
            "verdict": verdict,
        })

df = pd.DataFrame(records)
print(f"Collected {len(df)} trace verdicts across {df['run_name'].nunique()} runs")
df.head()

Collected 16 trace verdicts across 2 runs


,run_name,created_at,trace_id,dataset_item_id,verdict
0,2_metric_and_skill_fix,2026-05-22 00:06:00.402000+00:00,e4629c898a7de96745754b018808e944,f3f091b8-0e53-51d3-9e33-aaea1807f554,True
1,2_metric_and_skill_fix,2026-05-22 00:06:00.402000+00:00,a5dca171b07c53bfba52ebe2b3072445,64a94952-3c11-5c6c-8240-a02d9d2ef84a,True
2,2_metric_and_skill_fix,2026-05-22 00:06:00.402000+00:00,669623c5f5bb51cd0e40cd489f889728,c5482127-5edf-5ea5-8570-78dd616aa0f8,True
3,2_metric_and_skill_fix,2026-05-22 00:06:00.402000+00:00,ad2c422ce37c819078cf054dd5a5e456,3d4f391a-81f8-5459-b57b-513c3ec2bd09,True
4,2_metric_and_skill_fix,2026-05-22 00:06:00.402000+00:00,6992254307e92454de7905fc1cf990cc,899adfec-8b14-5147-9423-440930ae20ab,True


In [27]:
# Aggregate pass/fail per run
run_order = (
    df.groupby("run_name")["created_at"]
    .first()
    .sort_values()
    .index.tolist()
)

summary = (
    df.groupby("run_name")["verdict"]
    .agg(total="count", passed=lambda x: x.sum(), failed=lambda x: (~x).sum())
    .reindex(run_order)
)
summary["pass_rate"] = (summary["passed"] / summary["total"] * 100).round(1)
summary

,total,passed,failed,pass_rate
run_name,,,,
1_scaffold_baseline,8,5,3,62.5
2_metric_and_skill_fix,8,7,1,87.5


In [28]:
# Visualization: Pass Rate Trend + Pass/Fail Distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Pass Rate by Experiment Run", "Pass/Fail Distribution"),
    horizontal_spacing=0.12,
)

# Left panel: Pass rate trend line
fig.add_trace(
    go.Scatter(
        x=summary.index,
        y=summary["pass_rate"],
        mode="lines+markers+text",
        text=[f"{v}%" for v in summary["pass_rate"]],
        textposition="top center",
        textfont=dict(color="#2e7d32", size=12, family="Arial Black"),
        line=dict(color="#4caf50", width=3),
        marker=dict(size=10, color="#2e7d32"),
        showlegend=False,
    ),
    row=1, col=1,
)

# Right panel: Grouped bar chart
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["passed"],
        name="Pass",
        marker_color="#8bc34a",
        text=summary["passed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["failed"],
        name="Fail",
        marker_color="#ef9a9a",
        text=summary["failed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)

fig.update_yaxes(title_text="Pass Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(
    title_text=f"{DATASET_NAME.split('/')[-1]} Performance Analysis",
    title_x=0.5,
    title_font_size=18,
    barmode="group",
    height=450,
    width=1100,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.75),
    template="plotly_white",
)
fig.show()

In [29]:
# Per-Case Heatmap: identify which cases regress across experiments
if df["run_name"].nunique() > 1:
    pivot = df.pivot_table(
        index="dataset_item_id",
        columns="run_name",
        values="verdict",
        aggfunc="first",
    )
    pivot = pivot.reindex(columns=run_order)

    # Map item IDs to short labels
    item_labels = [f"item_{i+1}" for i in range(len(pivot))]

    heatmap = go.Figure(
        data=go.Heatmap(
            z=pivot.values.astype(float),
            x=pivot.columns.tolist(),
            y=item_labels,
            colorscale=[[0, "#ef9a9a"], [1, "#8bc34a"]],
            showscale=False,
            zmin=0,
            zmax=1,
            text=pivot.applymap(lambda v: "PASS" if v else "FAIL").values,
            texttemplate="%{text}",
            hovertemplate="Run: %{x}<br>Item: %{y}<br>Verdict: %{text}<extra></extra>",
        )
    )
    heatmap.update_layout(
        title="Per-Case Verdicts Across Experiments",
        title_x=0.5,
        xaxis_title="Experiment Run",
        yaxis_title="Dataset Item",
        height=max(300, len(pivot) * 35),
        width=900,
        template="plotly_white",
    )
    heatmap.show()
else:
    print("Heatmap requires at least 2 experiment runs for comparison.")

AttributeError: 'DataFrame' object has no attribute 'applymap'

In [32]:
import re
from IPython.display import display


def failure_drilldown(
    df,
    langfuse,
    filter_run=None,
    show_query=True,
    show_actual=False,
    show_expected=False,
):
    """Render a styled table of failed cases with metric details.

    Args:
        df: DataFrame with trace verdicts (from cell 5).
        langfuse: Langfuse client instance.
        filter_run: Optional run name to filter to.
        show_query: Include user_query column (trace input).
        show_actual: Include actual_output column.
        show_expected: Include expected_output column.
    """
    failed_traces = df[df["verdict"] == False]
    if filter_run:
        failed_traces = failed_traces[failed_traces["run_name"] == filter_run]

    if failed_traces.empty:
        print("No failed cases found.")
        return

    failure_details = []
    for _, row in failed_traces.iterrows():
        all_scores = langfuse.api.scores.get_many(trace_id=row["trace_id"], limit=50)
        trace = langfuse.api.trace.get(row["trace_id"])
        case_key = trace.metadata.get("case_key", "unknown") if trace.metadata else "unknown"
        case_name = trace.metadata.get("case_name", "") if trace.metadata else ""

        user_query = ""
        if show_query:
            user_query = trace.input or ""
        actual_output = ""
        if show_actual:
            actual_output = trace.output or ""
        expected_output = ""
        if show_expected:
            try:
                dataset_item = langfuse.api.dataset_items.get(row["dataset_item_id"])
                expected_output = dataset_item.expected_output or ""
            except Exception:
                expected_output = "(could not fetch)"

        for score in all_scores.data:
            if score.name == "verdict":
                continue
            is_failed = False
            if score.data_type == "BOOLEAN" and score.value == 0:
                is_failed = True
            elif score.data_type == "NUMERIC" and score.comment:
                if "passed=False" in score.comment:
                    is_failed = True

            if not is_failed:
                continue

            reasons_text = ""
            if score.comment and "\n\n" in score.comment:
                reasons_text = score.comment.split("\n\n", 1)[1]

            run_reasons = re.split(r"\n\n(?=\[Run \d+\])", reasons_text) if reasons_text else []

            entries = []
            if run_reasons:
                for run_reason in run_reasons:
                    entries.append(run_reason.strip())
            else:
                entries.append("(no reason recorded)")

            for reason in entries:
                record = {
                    "run": row["run_name"],
                    "case": f"{case_key}: {case_name}",
                    "metric": score.name,
                    "score": round(score.value, 3) if score.value is not None else None,
                    "reason": reason,
                }
                if show_query:
                    record["user_query"] = user_query
                if show_actual:
                    record["actual_output"] = actual_output
                if show_expected:
                    record["expected_output"] = expected_output
                failure_details.append(record)

    if not failure_details:
        print("Failed cases found but no individual metric failure details available.")
        return

    failures_df = pd.DataFrame(failure_details)
    print(f"{len(failures_df)} reason(s) across {failed_traces.shape[0]} failed case(s)\n")

    wide_cols = []
    if show_query:
        wide_cols.append("user_query")
    if show_actual:
        wide_cols.append("actual_output")
    if show_expected:
        wide_cols.append("expected_output")

    styled = failures_df.style.set_properties(**{
        "text-align": "left",
        "vertical-align": "top",
    })
    if wide_cols:
        styled = styled.set_properties(subset=wide_cols, **{
            "white-space": "pre-wrap",
            "font-family": "monospace",
            "font-size": "12px",
            "max-width": "500px",
            "overflow": "auto",
        })
    styled = (
        styled
        .set_properties(subset=["reason"], **{
            "white-space": "pre-wrap",
            "max-width": "300px",
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("vertical-align", "bottom")]},
        ])
        .hide(axis="index")
    )
    display(styled)

In [31]:
failure_drilldown(df, langfuse, show_actual=True, show_expected=True)

13 reason(s) across 4 failed case(s)



run,case,metric,score,reason,user_query,actual_output,expected_output
2_metric_and_skill_fix,case_3: Government project bullet notes → DPN entry with date conversion,Hallucination,0.667000,"[Run 1] The score is 1.00 because the actual output contradicts the context regarding the project duration, indicating a complete misalignment with the provided information.",Turn these bullet point notes into a proper DPN resume entry for my testing project at the Department of Social Services.,"Sure — since this is a DPN resume entry request, we should use the **DPN Resume Entry Drafter** skill so the output matches Deloitte People Network formatting rules (third-person, concise, correct date format, industry tagging, etc.).Here’s your DPN-formatted resume entry for the testing project at the Department of Social Services: **Department of Social Services - myGov Integration — Welfare Payments Portal Test Lead** 08/25 – 11/25 Government & Public Sector • Managed a team of six testers across onshore and offshore locations during the integration testing phase to ensure comprehensive coverage of API, UI and end-to-end scenarios. • Developed and executed an integration test strategy that addressed functional, non-functional and user acceptance requirements. • Coordinated defect triage with the development team, reducing the critical defect backlog by 60% within four weeks. • Implemented an automated regression suite using Selenium and Cucumber, achieving automation of 340 test cases to support continuous integration. **Skills & Technologies Used:** Selenium, Cucumber, Azure DevOps, SQL Server, Jenkins","A DPN resume entry with: bold header for DSS - myGov Integration, bold role Test Lead, dates converted from ""08-11/2025"" to 08/25 – 11/25, industry Government & Public Sector, exactly 4 bullet points in third person past tense, and 5 skills from the provided list. No personal pronouns. Australian English."
2_metric_and_skill_fix,case_3: Government project bullet notes → DPN entry with date conversion,Hallucination,0.667000,"[Run 2] The score is 1.00 because the actual output contains a contradiction regarding the project dates, stating them as 08/25 – 11/25, which does not align with the context that specifies the dates as 08-11/2025.",Turn these bullet point notes into a proper DPN resume entry for my testing project at the Department of Social Services.,"Sure — since this is a DPN resume entry request, we should use the **DPN Resume Entry Drafter** skill so the output matches Deloitte People Network formatting rules (third-person, concise, correct date format, industry tagging, etc.).Here’s your DPN-formatted resume entry for the testing project at the Department of Social Services: **Department of Social Services - myGov Integration — Welfare Payments Portal Test Lead** 08/25 – 11/25 Government & Public Sector • Managed a team of six testers across onshore and offshore locations during the integration testing phase to ensure comprehensive coverage of API, UI and end-to-end scenarios. • Developed and executed an integration test strategy that addressed functional, non-functional and user acceptance requirements. • Coordinated defect triage with the development team, reducing the critical defect backlog by 60% within four weeks. • Implemented an automated regression suite using Selenium and Cucumber, achieving automation of 340 test cases to support continuous integration. **Skills & Technologies Used:** Selenium, Cucumber, Azure DevOps, SQL Server, Jenkins","A DPN resume entry with: bold header for DSS - myGov Integration, bold role Test Lead, dates converted from ""08-11/2025"" to 08/25 – 11/25, industry Government & Public Sector, exactly 4 bullet points in third person past tense, and 5 skills from the provided list. No personal pronouns. Australian English."
1_scaffold_baseline,case_7: Minimal healthcare project description → DPN entry,Third Person Voice,0.333000,"[Run 1] The output contains personal pronouns such as 'we' and 'your', in